In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#读取数据
train_data=pd.read_csv('D:/Pytorch/project/data/Titanic Tutorial/raw/train.csv')
test_data=pd.read_csv('D:/Pytorch/project/data/Titanic Tutorial/raw/test.csv')

print(train_data.shape)
print(test_data.shape)

In [ ]:
#one-hot编码
features=['Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked']
X_train=train_data[features]
y_train=train_data['Survived']
X_test=test_data[features]

cat_features=['Name','Sex','Ticket','Cabin','Embarked']
X_train_cat=pd.get_dummies(X_train,columns=cat_features,dummy_na=True)
X_test_cat=pd.get_dummies(X_test,columns=cat_features,dummy_na=True)
X_test_cat=X_test_cat.reindex(columns=X_train_cat.columns,fill_value=0)

X_train_cat=X_train_cat.fillna(0).astype(np.float32)
X_test_cat=X_test_cat.fillna(0).astype(np.float32)

print(X_train_cat.shape)
print(X_test_cat.shape)


In [ ]:
#转成张量
X_train_tor=torch.tensor(X_train_cat.values,dtype=torch.float32)
X_test_tor=torch.tensor(X_test_cat.values,dtype=torch.float32)
y_train_tor=torch.tensor(y_train.values,dtype=torch.long)

print(X_train_tor.shape)
print(X_test_tor.shape)

In [ ]:
#超参数
lr=0.001
num_epochs=100
batch_size=256

In [ ]:
dataset=TensorDataset(X_train_tor,y_train_tor)
X_train_load=DataLoader(dataset,batch_size=batch_size,shuffle=True)
X_test_load=DataLoader(X_test_tor,batch_size=batch_size,shuffle=False)

In [ ]:
#构建网络
in_features=X_train_tor.shape[1]
net=nn.Sequential(nn.Linear(in_features,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU(),
                  nn.Linear(256,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),
                  nn.Linear(64,2)).to(DEVICE)

In [ ]:
#损失函数和优化器
loss=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(net.parameters(),lr=lr)

In [ ]:
#训练循环
for epoch in range(num_epochs):
    net.train()
    total_loss=0
    total_acc=0
    n=0

    for X,y in X_train_load:
        X,y=X.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad()
        y_hat=net(X)
        l=loss(y_hat,y)
        l.backward()
        optimizer.step()

        total_loss+=l.item()*y.shape[0]
        total_acc+=(y_hat.argmax(dim=1)==y).sum().item()
        n+=y.shape[0]
        print(f'epoch {epoch+1},loss {total_loss/n:.4f},acc {total_acc/n:.4f}')


In [ ]:
#预测
net.eval()
with torch.no_grad():
    y_test_hat=net(X_test_tor.to(DEVICE))
    y_pred=y_test_hat.argmax(dim=1).cpu().numpy()

In [ ]:
#提交csv文件
submission=pd.DataFrame({
    'PassengerId':test_data['PassengerId'],
    'Survived':y_pred.astype(int)
})
submission.to_csv('submission002.csv',index=False)
print(submission.head())